# BERT - 트랜스포머 인코더 기반 언어 이해 모델

- 이전 모델들이 한 방향으로만 단어를 이해한 것과 달리, 단어의 양방향 문맥을 동시에 학습함

- 대규모 텍스트 데이터를 활용해 사전 학습을 수행한 뒤, 다양한 자연어 처리 작업에 쉽게 적용할 수 있는 범용 모델로 설계됨

- BooksCorpus 와 위키피디아 데이터를 사용해 사전 훈련되었으며, 두 가지 사전 훈련 방식을 사용
    - Masked Language Model(MLM)
        - 입력 데이터의 일부 토큰을 가린(masking) 다음, 모델이 가려진 단어를 예측

    - Next Sentence Prediction(NSP)
        - 두 문장이 제시됐을 때 두 번째 문장이 첫 번째 문장에 이어지는 다음 문장인지를 예측

- 문맥을 이해하는 능력이 뛰어나 다양한 자연어 처리 작업에서 활용

- 학습된 출력 벡터를 기반으로 수행되는 별도의 분류기를 제외한 나머지 부분을 백본(backbone) 또는 베이스(base)라고 부름
    - BERT는 여러 개의 트랜스포머 인코더 블록을 포함하고 있는 BERT 백본과 분류기로 구성됨

- 입력 데이터
    - 세 가지 임베딩을 더해 구성됨
        - 토큰 임베딩
        - 위치 임베딩
        - 세그먼트 임베딩
            - 다음 문장 예측 작업을 위해 두 개의 문장을 0과 1로 구분
    - 세 개의 임베딩을 모두 더해 층 정규화와 드롭아웃 층을 통과시킨 후 트랜스포머 인코더 입력으로 전달됨

- 트랜스포머 인코더 블록
    - BERT 백본의 핵심은 반복적으로 쌓아 올린 트랜스포머 인코더

In [1]:
# 훈련 데이터를 위한 datasets, 모델 평가를 위한 evaluate 라이브러리 설치
!pip install datasets evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.9 MB/s eta 0:00:00


In [2]:
import keras_nlp
import keras
from keras import layers
import matplotlib.pyplot as plt
import pandas as pd
from transformers import AutoModelForSequenceClassification, AutoTokenizer, TrainingArguments, Trainer
import evaluate
from datasets import Dataset
import numpy as np

# KerasNLP 로 영화 리뷰 텍스트의 감성 분류

## KerasNLP로 BERT 모델 로드

- IMDB 데이터셋
    - train 폴더와 test 폴더 안에는 각각 neg 폴더와 pos 폴더가 있음

    - neg 폴더에는 부정적인 감성의 텍스트

    - pos 폴더에는 긍정적인 감성의 텍스트가 포함

In [3]:
!unzip /content/aclImdb.zip

스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
  inflating: aclImdb/train/pos/5504_10.txt  
  inflating: aclImdb/train/pos/5505_10.txt  
  inflating: aclImdb/train/pos/5506_10.txt  
  inflating: aclImdb/train/pos/5507_7.txt  
  inflating: aclImdb/train/pos/5508_10.txt  
  inflating: aclImdb/train/pos/5509_10.txt  
  inflating: aclImdb/train/pos/550_10.txt  
  inflating: aclImdb/train/pos/5510_10.txt  
  inflating: aclImdb/train/pos/5511_8.txt  
  inflating: aclImdb/train/pos/5512_7.txt  
  inflating: aclImdb/train/pos/5513_7.txt  
  inflating: aclImdb/train/pos/5514_9.txt  
  inflating: aclImdb/train/pos/5515_7.txt  
  inflating: aclImdb/train/pos/5516_7.txt  
  inflating: aclImdb/train/pos/5517_9.txt  
  inflating: aclImdb/train/pos/5518_8.txt  
  inflating: aclImdb/train/pos/5519_9.txt  
  inflating: aclImdb/train/pos/551_8.txt  
  inflating: aclImdb/train/pos/5520_9.txt  
  inflating: aclImdb/train/pos/5521_8.txt  
  inflating: aclImdb/train/pos/5522_8.txt  
  inflating: aclImdb/train/pos/5523

In [4]:
# 폴더에 있는 데이터를 읽어들임
train_ds, val_ds = keras.utils.text_dataset_from_directory(
    "aclImdb/train", subset = "both", validation_split = 0.2, seed = 42)

Found 25000 files belonging to 2 classes.
Using 20000 files for training.
Using 5000 files for validation.


In [5]:
test_ds = keras.utils.text_dataset_from_directory("aclImdb/test")

Found 25000 files belonging to 2 classes.


In [6]:
# train_ds 에서 샘플 텍스트 추출
# 기본적으로 text_dataset_from_directory() 함수는 크기가 32인 배치 데이터셋을 만들기 때문에
# unbatch() 함수로 배치를 해제한 다음 take(1) 을 호출하여 첫 번째 원소를 가져옴
feature, target = train_ds.unbatch().take(1).get_single_element()

In [7]:
feature.numpy()

b'"Pandemonium" is a horror movie spoof that comes off more stupid than funny. Believe me when I tell you, I love comedies. Especially comedy spoofs. "Airplane", "The Naked Gun" trilogy, "Blazing Saddles", "High Anxiety", and "Spaceballs" are some of my favorite comedies that spoof a particular genre. "Pandemonium" is not up there with those films. Most of the scenes in this movie had me sitting there in stunned silence because the movie wasn\'t all that funny. There are a few laughs in the film, but when you watch a comedy, you expect to laugh a lot more than a few times and that\'s all this film has going for it. Geez, "Scream" had more laughs than this film and that was more of a horror film. How bizarre is that?<br /><br />*1/2 (out of four)'

In [8]:
target.numpy()

np.int32(0)

In [9]:
# bert 백본과 토크나이저, 분류기를 합친 BERT 모델 로드
# num_classes : 2 -> 긍정과 부정으로 분류하기 위해 분류할 클래스 개수를 2로 지정
# en : 영어 텍스트 말뭉치에서 훈련된 모델
# uncased : 대소문자를 구분하지 않음
classifier = keras_nlp.models.BertClassifier.from_preset("bert_tiny_en_uncased", num_classes = 2)

100%|██████████| 454/454 [00:00<00:00, 1.03MB/s]


In [10]:
classifier.summary()

Preprocessor: "bert_text_classifier_preprocessor"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                                                  ┃                                   Config ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ bert_tokenizer (BertTokenizer)                                │                       Vocab size: 30,522 │
└───────────────────────────────────────────────────────────────┴──────────────────────────────────────────┘

Model: "bert_text_classifier"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ padding_mask (InputLayer)     │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ segment_ids (InputLayer)      │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_ids (InputLayer)        │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ bert_backbone (BertBackbone)  │ [(None, 128), (None,      │       4,385,920 │ padding_mask[0][0],        │
│                               │ None, 128)]               │                 │ segment_ids[0][0],         │
│                               │                           │                 │ token_ids[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ classifier_dropout (Dropout)  │ (None, 128)               │               0 │ bert_backbone[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ logits (Dense)                │ (None, 2)                 │             258 │ classifier_dropout[0][0]   │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 4,386,178 (16.73 MB)

 Trainable params: 4,386,178 (16.73 MB)

 Non-trainable params: 0 (0.00 B)

- 읽어들인 BERT 모델은 사전훈련된 가중치가 자동으로 다운로드 됨

- bert_tiny 모델은 BERT 모델 중 가장 작은 모델이지만 파라미터의 개수는 4백만 개가 넘음

- num_classes 를 2로 설정했기 때문에 출력층의 유닛은 2로 설정됨

## BERT 모델 파인 튜닝

- 기본적으로 BertClassifier 에 포함된 백본 모델은 훈련이 가능함
    - classifier 를 훈련할 때 백본 모델도 함께 훈련됨
    - 백본 모델의 가중치를 훈련하고 싶지 않다면 classifier.backbone.trainable = False 로 지정

In [11]:
classifier.fit(train_ds, validation_data = val_ds, epochs = 5)

Epoch 1/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 82s 73ms/step - loss: 0.5789 - sparse_categorical_accuracy: 0.6827 - val_loss: 0.3542 - val_sparse_categorical_accuracy: 0.8478
Epoch 2/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 40s 63ms/step - loss: 0.3264 - sparse_categorical_accuracy: 0.8624 - val_loss: 0.3014 - val_sparse_categorical_accuracy: 0.8738
Epoch 3/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 40s 63ms/step - loss: 0.2575 - sparse_categorical_accuracy: 0.8972 - val_loss: 0.2894 - val_sparse_categorical_accuracy: 0.8800
Epoch 4/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 41s 65ms/step - loss: 0.1994 - sparse_categorical_accuracy: 0.9284 - val_loss: 0.3239 - val_sparse_categorical_accuracy: 0.8744
Epoch 5/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 41s 65ms/step - loss: 0.1491 - sparse_categorical_accuracy: 0.9470 - val_loss: 0.3278 - val_sparse_categorical_accuracy: 0.8774


In [12]:
# 예측
classifier.predict([feature])

1/1 ━━━━━━━━━━━━━━━━━━━━ 6s 6s/step


array([[ 2.0453217, -2.303829 ]], dtype=float32)

In [13]:
target.numpy()

np.int32(0)

In [14]:
# 파인 튜닝한 bert_tiny 모델의 테스트 세트 정확도 확인
classifier.evaluate(test_ds)

782/782 ━━━━━━━━━━━━━━━━━━━━ 18s 21ms/step - loss: 0.3821 - sparse_categorical_accuracy: 0.8620


[0.3778691291809082, 0.8625199794769287]

# 허깅페이스로 영화 리뷰 텍스트 감성 분류

- 허깅페이스 : 인공지능 모델을 쉽게 사용하고 공유할 수 있게 해주는 플랫폼 + 오픈소스 생태계

- transformers 라이브러리 : 허깅페이스에서 제공하는 딥러닝 기반의 자연어 처리 라이브러리

## 네이버 영화 리뷰 데이터셋 준비

- nsmc : IMDB 영화 리뷰 데이터셋과 비슷하게 네이버 영화 리뷰 댓글을 수집한 한글 데이터셋

In [19]:
nsmc_train = pd.read_csv("ratings_train.txt", sep = "\t")

In [20]:
nsmc_train.head()

,id,document,label
0,9976970,아 더빙.. 진짜 짜증나네요 목소리,0
1,3819312,흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나,1
2,10265843,너무재밓었다그래서보는것을추천한다,0
3,9045019,교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정,0
4,6483659,사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 ...,1


In [21]:
nsmc_test = pd.read_csv("ratings_test.txt", sep = "\t")

In [22]:
nsmc_test.head()

,id,document,label
0,6270596,굳 ㅋ,1
1,9274899,GDNTOPCLASSINTHECLUB,0
2,8544678,뭐야 이 평점들은.... 나쁘진 않지만 10점 짜리는 더더욱 아니잖아,0
3,6825595,지루하지는 않은데 완전 막장임... 돈주고 보기에는....,0
4,6723715,3D만 아니었어도 별 다섯 개 줬을텐데.. 왜 3D로 나와서 제 심기를 불편하게 하죠??,0


In [23]:
nsmc_train.shape

(150000, 3)

In [24]:
nsmc_test.shape

(50000, 3)

In [38]:
nsmc_train.isna().sum()

,0
id,0
document,5
label,0


In [41]:
nsmc_train = nsmc_train.dropna()
nsmc_test = nsmc_test.dropna()

In [42]:
train_dataset = Dataset.from_pandas(nsmc_train)
test_dataset = Dataset.from_pandas(nsmc_test)

In [43]:
train_dataset

Dataset({
    features: ['id', 'document', 'label', '__index_level_0__'],
    num_rows: 149995
})

## 백본 모델 준비

- 허깅페이스에는 매우 많은 모델이 있기 때문에 모든 모델이 독자적인 클래스로 제공되지 않는 대신 작업마다 공통으로 사용할 수 있는 클래스인 Auto클래스를 제공
    - 텍스트 분류에 해당하는 Auto 클래스가 AutoModelForSequenceClassification

In [44]:
bert_kor = AutoModelForSequenceClassification.from_pretrained(
    "bongsoo/bert-small-kor-v1", num_labels = 2
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bongsoo/bert-small-kor-v1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [45]:
print(bert_kor)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(10022, 512, padding_idx=0)
      (position_embeddings): Embedding(512, 512)
      (token_type_embeddings): Embedding(2, 512)
      (LayerNorm): LayerNorm((512,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-3): 4 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=512, out_features=512, bias=True)
              (key): Linear(in_features=512, out_features=512, bias=True)
              (value): Linear(in_features=512, out_features=512, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=512, out_features=512, bias=True)
              (LayerNorm): LayerNorm((512,), eps=1e-1

## 입력 데이터 토큰화

- BERT 모델은 토큰화를 위해 토큰 아이디와 세그먼트 아이디, 패딩 마스크가 필요

- transformers 라이브러리는 토크나이저도 모델과 비슷한 방식으로 로드함

In [46]:
# 허깅페이스에서 제공하는 자동 토크나이저 클래스
bert_kor_tokenizer = AutoTokenizer.from_pretrained(
    "bongsoo/bert-small-kor-v1"
)

In [47]:
train_dataset

Dataset({
    features: ['id', 'document', 'label', '__index_level_0__'],
    num_rows: 149995
})

In [48]:
# 훈련 데이터 샘플 텍스트 변환
print(train_dataset[0]["document"])
prep_data = bert_kor_tokenizer(train_dataset[0]["document"])
print(prep_data)

아 더빙.. 진짜 짜증나네요 목소리
{'input_ids': [2, 606, 261, 1519, 17, 17, 4668, 766, 1400, 1132, 1464, 1130, 2889, 3], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [49]:
# 토큰 아이디를 토큰으로 변환
tokens = bert_kor_tokenizer.convert_ids_to_tokens(prep_data["input_ids"])
print(tokens)

['[CLS]', '아', '더', '##빙', '.', '.', '진짜', '짜', '##증', '##나', '##네', '##요', '목소리', '[SEP]']


In [50]:
# 원본 문자열로 변환도 가능
bert_kor_tokenizer.convert_tokens_to_string(tokens)

'[CLS] 아 더빙.. 진짜 짜증나네요 목소리 [SEP]'

In [51]:
# 배치 데이터가 전달되면 bert_kor_tokenizer를 호출하여 문자열을 전처리하는 함수
def tokenize(batch):
    return bert_kor_tokenizer(batch["document"], padding = True, truncation = True)

In [52]:
nsmc_tokenized_train = train_dataset.map(tokenize, batched = True, batch_size = None)
nsmc_tokenized_test = test_dataset.map(tokenize, batched = True, batch_size = None)

Map:   0%|          | 0/149995 [00:00<?, ? examples/s]

Map:   0%|          | 0/49997 [00:00<?, ? examples/s]

In [53]:
# 모든 데이터를 다 훈련하면 시간이 매우 오래 걸려서 일부 샘플링
nsmc_train = nsmc_tokenized_train.shuffle(seed = 42).select(range(1000))
nsmc_test = nsmc_tokenized_test.shuffle(seed = 42).select(range(100))

## BERT 모델 파인 튜닝

- transformers 라이브러리로 모델을 훈련하면 모델의 성능을 평가하는 함수를 정의해서 전달해야함

- evaluate 패키지를 통해 정확도 지표를 로드

In [54]:
metric = evaluate.load("accuracy")

In [55]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis = -1)
    return metric.compute(predictions = predictions, references = labels)

In [56]:
training_args = TrainingArguments(output_dir = "bert_kor_nsmc",
                                  num_train_epochs = 5,
                                  eval_strategy = "epoch",
                                  save_strategy = "epoch",
                                  logging_steps = len(nsmc_train) // 8,
                                  load_best_model_at_end = True,
                                  report_to = "none")

In [58]:
trainer = Trainer(model = bert_kor,
                  train_dataset = nsmc_train,
                  eval_dataset = nsmc_test,
                  args = training_args,
                  compute_metrics = compute_metrics)

In [59]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.608300,0.494849,0.750000
2,0.424600,0.479480,0.780000
3,0.271000,0.534817,0.800000
4,0.150500,0.772358,0.790000
5,0.120600,0.716382,0.800000


TrainOutput(global_step=625, training_loss=0.31500886993408206, metrics={'train_runtime': 52.3571, 'train_samples_per_second': 95.498, 'train_steps_per_second': 11.937, 'total_flos': 54844270920000.0, 'train_loss': 0.31500886993408206, 'epoch': 5.0})

In [60]:
# 사전 훈련된 BERT 모델로 예측
preds_output = trainer.predict(nsmc_test)

In [61]:
preds_output.label_ids[:5]

array([0, 0, 1, 1, 0])

In [62]:
from transformers import pipeline

sentiment_model = pipeline(model="WhitePeak/bert-base-cased-Korean-sentiment")

config.json:   0%|          | 0.00/895 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/711M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/711M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/367 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Device set to use cuda:0


NameError: name 'sentiment_mode' is not defined

In [63]:
sentiment_model("매우 좋아")

[{'label': 'LABEL_1', 'score': 0.8098211288452148}]

In [64]:
sentiment_model("별루")

[{'label': 'LABEL_0', 'score': 0.9970722198486328}]

In [65]:
sentiment_model("잘도 그러겠다")

[{'label': 'LABEL_0', 'score': 0.9111682176589966}]